Autor: JESUS ALEJANDRO COLIN VILCHIS

target:
  Analysis of SARS-Cov-2 sequence from Mexican patients from GISAID database, 
from March 2020 to July 2021.

Libraries are imported

In [ ]:
!pip install --upgrade plotly
!pip install pandas
!pip install numpy
import pandas as pd
import plotly.express as px
import re
import numpy as np


Reading the metadata

In [ ]:
path='Copia de delfin_agosto_AMC_v3.xlsx'
df=pd.read_excel(path)

Initiating the variables to use

In [ ]:
protein=['Spike_','E_','N_','M_','NSP']
Parts ={
'Spike':[
          ['S1A domain',1,302],
          ['S1A-S1B linker',303,332],
          ['S1B domain',333,527],
          ['S1B - S1C linker',528,533],
          ['S1C domain',534,589],
          ['S1C - S1D linker',590,593],
          ['S1D domain',594,674],
          ['Protease cleavage site',675,692],
          ['S1-S2 subunits linker',693,710],
          ['Central β-strand',711,737],
          ['Downward helix',738,782],
          ['S2’ cleavage site',783,815],
          ['Fusion peptide',816,828],
          ['Connecting region',829,911],
          ['Heptad repeat region 1',912,983],
          ['Central helix',984,1034],
          ['β-hairpin',1035,1068],
          ['β-sheet domain',1069,1133],
          ['Heptad repeat region 2',1134,1213],
          ['Transmembrane region',1214,1236],
          ['Cytoplasmic region',1237,1273],
], #Guruprasad L. Human SARS-CoV-2 spike protein mutations. Proteins. 2021 May;89(5):569-576. doi: 10.1002/prot.26042. Epub 2021 Jan 17. PMID: 33423311; PMCID: PMC8014176
'M':[
      ['N – Terminal (NTD)',1,19],
      ['TMl',20,40],
      ['Mnone',41,50],
      ['TMll',51,71],
      ['Mnone',72,79],
      ['TMlll',80,100],
      ['C-Terminal (CTD)',101,222],
 ],
'N':[
      ['N-terminal (NTD)',1,50],
      ['Dominio de unión al ARN (RBD)',51,174],
      ['Enlazador central desordenado predicho (LINK)',175,246],
      ['Dominio de dimerización',247,365],
      ['C-terminal (CTD)',366,419],
 ],
 'E':[
      ['N-termina',1,16],
      ['Transmembranal (TMD)',17,37],
      ['C-Terminal',36,76]
 ]
 }


the number of total mutations are counted

In [ ]:
# give me the aminoacid mutation and the segment where it occurred for each user of the database, for each mutated aminoacid after cleaning it, for each protein segment only when the segment is in the mutated aminoacid
amino={'Spike_':[],'E_':[],'N_':[],'M_':[],'NSP':[]}
for user in df['AA Substitutions'].tolist():
  for AA in user.replace(r'(','').replace(r')','').split(','):
    for segment in protein:
      if re.search(segment,AA) is not None:
        amino[segment].append(AA)
        
for segment in amino.keys():
  print(segment,' : ',len(amino[segment]))

for segment in protein:
  value, counts = np.unique(amino[segment], return_counts=True)
  amino[segment]=[(value[i],counts[i],value[i].split('_')[0],value[i].split('_')[1],int(re.findall("\d+",value[i].split('_')[1])[0])) for i in range(len(value))]
  

the variety of possible mutations are counted

In [ ]:
for segment in amino.keys():
  print(segment,' : ',len(amino[segment]))
  

The **tables of the regions** are created, but we only show the one for the Spike protein, ***you can visualize some others by changing the 0 to 1, 2 or 3***

In [ ]:
myDF=[]
for segment in amino.keys():
  myDF.append(pd.DataFrame(amino[segment],columns=[segment,'count','full','change','position'])) 
for i in range(len(myDF)-1):
  myDF[i]['area'] = [x[0] for num in myDF[i].position for x in Parts[myDF[i].full[0]] if num >= x[1] and num <= x[2]]
myDF[0]

The NSP's table is a special case, but it is elaborated in this section

In [ ]:
for i in range(len(myDF)-1):
  myDF[i]['area'] = [x[0] for num in myDF[i].position for x in Parts[myDF[i].full[0]] if num >= x[1] and num <= x[2]]
myDF[4]['area'] = [int(re.findall("\d+",x)[0]) for x in myDF[4].full]
myDF[4]

# Results **display**


In [ ]:
fig =px.sunburst(
    myDF[0],
    path=['area','position','Spike_'],
    values='count',
    title="Spike protein pie graph",
)
fig.show()

In [ ]:
fig =px.sunburst(
    myDF[1],
    path=['area','position','E_'],
    values='count',
    title="E protein pie graph",
)
fig.show()

In [ ]:
fig =px.sunburst(
    myDF[2],
    path=['area','position','N_'],
    values='count',
    title="N protein pie graph")
fig.show()

In [ ]:
fig =px.sunburst(
    myDF[3],
    path=['area','position','M_'],
    values='count',
    title="M protein pie graph",
)
fig.show()

In [ ]:
fig =px.sunburst(
    myDF[4],
    path=['area','position','NSP'],
    values='count',
    title="NSP protein pie graph",
)
fig.show()

In [ ]:
dic =[mutation for num,segment in enumerate(amino.keys()) for mutation in myDF[num][segment]]

In [ ]:
dicS=[mutation for mutation in myDF[0]['Spike_']]
dicE=[mutation for mutation in myDF[1]['E_']]
dicN=[mutation for mutation in myDF[2]['N_']]
dicM=[mutation for mutation in myDF[3]['M_']]
dicNSP=[mutation for mutation in myDF[4]['NSP']]

In [ ]:
record=[]
val=0
#An iteration is generated for each day of the week up to the maximum
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  # Existing mutations are iterated
  for mutation in dicS:
    val=0
    # Columns are filtered for each week
    for user in df[df.sem_cont == week]['AA Substitutions']:
      # we do a count when the mutation appears in the user
      if mutation in user:
        val=val+1
    row.append(val)
  record.append(row)
# to corroborate the results, the protein S dataframe is printed
dfS=pd.DataFrame(record,columns=dicS)
fig = px.line(dfS, 
              x=range(1,len(dfS)+1), 
              y=dfS.columns,
              title="Spike protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
dfS.head()

In [ ]:
#CAMBIAR SEMANA 2
record=[]
val=0
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  for mutation in dicM:
    val=0
    for user in df[df.sem_cont == week]['AA Substitutions']:
      if mutation in user:
        val=val+1
    row.append(val)      
  record.append(row)
dfM=pd.DataFrame(record,columns=dicM)
fig = px.line(dfM, 
              x=range(1,len(dfM)+1), 
              y=dfM.columns,
              title="M protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
record=[]
val=0
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  for mutation in dicN:
    val=0
    for user in df[df.sem_cont == week]['AA Substitutions']:
      if mutation in user:
        val=val+1
    row.append(val)
      
  record.append(row)
  

#Writing the table to the dataframe structure
dfN=pd.DataFrame(record,columns=dicN)

# organizing the chart
fig = px.line(dfN, 
              x=range(1,len(dfN)+1), 
              y=dfN.columns,
              title="N protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
record=[]
val=0
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  for mutation in dicE:
    val=0
    for user in df[df.sem_cont == week]['AA Substitutions']:
      if mutation in user:
        val=val+1
    row.append(val)
  record.append(row)
  

dfE = pd.DataFrame(record,columns=dicE)
fig = px.line(dfE, 
              x=range(1,len(dfE)+1), 
              y=dfE.columns,
              title="E protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
record=[]
val=0
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  for mutation in dicNSP:
    val=0
    for user in df[df.sem_cont == week]['AA Substitutions']:
      if mutation in user:
        val=val+1
    row.append(val)
  record.append(row)
  

dfNSP = pd.DataFrame(record,columns=dicNSP)
fig = px.line(dfNSP, 
              x=range(1,len(dfNSP)+1), 
              y=dfNSP.columns,
              title="NSP protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
record=[]
val=0
for week in range(1,int(df.sem_cont.max())+1):
  row=[]
  for mutation in dic:
    val=0
    for user in df[df.sem_cont == week]['AA Substitutions']:
      if mutation in user:
        val=val+1
    row.append(val)
  record.append(row)


In [ ]:
df1=pd.DataFrame(record,columns=dic)
fig = px.line(df1, 
              x=range(1,len(df1)+1), 
              y=df1.columns,
              title="Full protein time-line",)
fig.update_layout(
    xaxis_title="Weeks elapsed",
    yaxis_title="Number of repetitions",
    legend_title="Mutations",
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="RebeccaPurple"
    )
)

fig.show()